# Transformer Encoder for NLE ML Replacement (Iteration 2)
This notebook trains a Transformer Encoder to replace a conventional DSP Nonlinear Equalizer (NLE). 
We perform delta learning (residual prediction) to correct the distorted `NLE OFF` signal towards the `NLE ON` reference.
This iteration introduces architectural improvements (`norm_first=True`), a dynamic QPSK snapping loss, and stabilized amplitude tracking.


## 1. Imports

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import copy
import math
import os
import pandas as pd


## 2. Configuration

In [ ]:
# Model Config V2
WINDOW = 256
STRIDE = 128
D_MODEL = 128
NHEAD = 4
NUM_LAYERS = 3
DIM_FEEDFORWARD = 256
DROPOUT = 0.1
USE_POSITIONAL_ENCODING = True

# Training Config
BATCH_SIZE = 32
EPOCHS = 150
LR = 3e-4
WEIGHT_DECAY = 1e-4
VAL_SPLIT = 0.2
EARLY_STOP_PATIENCE = 20

# Loss Weights V2
AMP_WEIGHT = 0.5
SNAP_WEIGHT = 0.01
DC_WEIGHT = 0.1
EPSILON = 1e-8


## 3. Device and Reproducibility

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()


## 4. Load Data and Check Normalization
We load the NLE OFF (input) and NLE ON (reference) data and print their statistics before applying joint normalization.

In [ ]:
off_data = np.load('data/nle_off_10k.npy')
on_data = np.load('data/nle_on_10k.npy')

def print_stats(name, data):
    rms = np.sqrt(np.mean(data**2))
    print(f"--- {name} Stats ---")
    print(f"Min: {np.min(data, axis=0)}")
    print(f"Max: {np.max(data, axis=0)}")
    print(f"Mean: {np.mean(data, axis=0)}")
    print(f"Std: {np.std(data, axis=0)}")
    print(f"RMS Magnitude: {rms:.4f}
")

print_stats("NLE OFF", off_data)
print_stats("DSP NLE ON", on_data)

# Joint Normalization to preserve Delta relationship
# We compute mean and std from the NLE OFF data, but apply it to BOTH to maintain relative scale.
mean_val = np.mean(off_data, axis=0)
std_val = np.std(off_data, axis=0)

off_data_norm = (off_data - mean_val) / std_val
on_data_norm = (on_data - mean_val) / std_val

print("After Joint Normalization:")
print_stats("NLE OFF (Norm)", off_data_norm)
print_stats("DSP NLE ON (Norm)", on_data_norm)


## 5. Clean Train/Validation Split
To avoid data leakage, we split the continuous signal first before creating overlapping windows.

In [ ]:
total_samples = len(off_data_norm)
train_size = int(total_samples * (1 - VAL_SPLIT))

train_off = off_data_norm[:train_size]
train_on = on_data_norm[:train_size]

val_off = off_data_norm[train_size:]
val_on = on_data_norm[train_size:]

print("Train split size:", len(train_off))
print("Val split size:", len(val_off))


## 6. Sequence Window Dataset
Generate overlapping sequence windows.

In [ ]:
class NLEWindowDataset(Dataset):
    def __init__(self, x_data, y_data, window, stride):
        self.x_windows = []
        self.delta_windows = []
        
        delta_data = y_data - x_data
        num_windows = (len(x_data) - window) // stride + 1
        for i in range(num_windows):
            start = i * stride
            end = start + window
            self.x_windows.append(x_data[start:end])
            self.delta_windows.append(delta_data[start:end])
            
    def __len__(self):
        return len(self.x_windows)
        
    def __getitem__(self, idx):
        return torch.tensor(self.x_windows[idx], dtype=torch.float32),                torch.tensor(self.delta_windows[idx], dtype=torch.float32)

train_dataset = NLEWindowDataset(train_off, train_on, WINDOW, STRIDE)
val_dataset = NLEWindowDataset(val_off, val_on, WINDOW, STRIDE)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")


## 7. Transformer Model V2
Using `norm_first=True` for optimization stability and a toggle for Positional Encoding ablation.

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

class TransformerNLE_v2(nn.Module):
    def __init__(self, d_model, nhead, num_layers, dim_feedforward, dropout, use_pe):
        super().__init__()
        self.use_pe = use_pe
        self.input_linear = nn.Linear(2, d_model)
        if self.use_pe:
            self.pos_encoder = PositionalEncoding(d_model)
            
        # Try using norm_first=True. If PyTorch is older, it might fallback or fail. 
        # Modern PyTorch supports it.
        try:
            encoder_layers = nn.TransformerEncoderLayer(
                d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward, 
                dropout=dropout, batch_first=True, norm_first=True
            )
        except TypeError:
            print("norm_first=True not supported by your PyTorch version, falling back to default.")
            encoder_layers = nn.TransformerEncoderLayer(
                d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward, 
                dropout=dropout, batch_first=True
            )
            
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers=num_layers)
        self.output_linear = nn.Linear(d_model, 2)
        
    def forward(self, src):
        x = self.input_linear(src)
        if self.use_pe:
            x = self.pos_encoder(x)
        x = self.transformer_encoder(x)
        out = self.output_linear(x)
        return out

model = TransformerNLE_v2(D_MODEL, NHEAD, NUM_LAYERS, DIM_FEEDFORWARD, DROPOUT, USE_POSITIONAL_ENCODING).to(device)
print(model)


## 8. Improved Dynamic Loss Function
Dynamic QPSK radius estimation from the batch reference and stabilized amplitude loss.

In [ ]:
def domain_aware_loss_v2(pred_delta, true_delta, x):
    # 1. IQ Correction MSE
    mse_iq = torch.mean((pred_delta - true_delta)**2)
    
    y_hat = x + pred_delta
    y_true = x + true_delta
    
    # 2. Amplitude Preservation (Stabilized)
    amp_hat = torch.sqrt(y_hat[..., 0]**2 + y_hat[..., 1]**2 + EPSILON)
    amp_true = torch.sqrt(y_true[..., 0]**2 + y_true[..., 1]**2 + EPSILON)
    mse_amp = torch.mean((amp_hat - amp_true)**2)
    
    # 3. Dynamic QPSK Snapping
    # Estimate the ideal QPSK radius dynamically from the true reference
    dynamic_radius = torch.mean(amp_true)
    ideal_mag = dynamic_radius.item() / math.sqrt(2) # Projections on I/Q axes
    
    ideal_points = torch.tensor([
        [-ideal_mag, -ideal_mag], [-ideal_mag, ideal_mag],
        [ideal_mag, -ideal_mag], [ideal_mag, ideal_mag]
    ], device=x.device)
    
    distances = torch.sum((y_hat.unsqueeze(2) - ideal_points.view(1, 1, 4, 2))**2, dim=-1)
    min_distances, _ = torch.min(distances, dim=-1)
    snap_loss = torch.mean(min_distances)
    
    # 4. DC Centering
    mean_I = torch.mean(y_hat[..., 0])
    mean_Q = torch.mean(y_hat[..., 1])
    dc_loss = mean_I**2 + mean_Q**2
    
    total_loss = mse_iq + AMP_WEIGHT * mse_amp + SNAP_WEIGHT * snap_loss + DC_WEIGHT * dc_loss
    
    return total_loss, mse_iq.item(), mse_amp.item(), snap_loss.item(), dc_loss.item()


## 9. Optimizer and Scheduler

In [ ]:
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)


## 10. Training Loop with Gradient Clipping

In [ ]:
history = {
    'train_loss': [], 'val_loss': [],
    'train_mse_iq': [], 'train_mse_amp': [], 'train_snap': [], 'train_dc': []
}

best_val_loss = float('inf')
patience_counter = 0

for epoch in range(EPOCHS):
    model.train()
    epoch_metrics = np.zeros(5) # total, iq, amp, snap, dc
    
    for x_batch, delta_batch in train_loader:
        x_batch, delta_batch = x_batch.to(device), delta_batch.to(device)
        
        optimizer.zero_grad()
        pred_delta = model(x_batch)
        total_loss, mse_iq, mse_amp, snap, dc = domain_aware_loss_v2(pred_delta, delta_batch, x_batch)
        
        total_loss.backward()
        # Gradient Clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        epoch_metrics += np.array([total_loss.item(), mse_iq, mse_amp, snap, dc])
        
    scheduler.step()
    epoch_metrics /= len(train_loader)
    
    history['train_loss'].append(epoch_metrics[0])
    history['train_mse_iq'].append(epoch_metrics[1])
    history['train_mse_amp'].append(epoch_metrics[2])
    history['train_snap'].append(epoch_metrics[3])
    history['train_dc'].append(epoch_metrics[4])
    
    # Validation
    model.eval()
    epoch_val_loss = 0.0
    with torch.no_grad():
        for x_batch, delta_batch in val_loader:
            x_batch, delta_batch = x_batch.to(device), delta_batch.to(device)
            pred_delta = model(x_batch)
            loss_val, _, _, _, _ = domain_aware_loss_v2(pred_delta, delta_batch, x_batch)
            epoch_val_loss += loss_val.item()
            
    epoch_val_loss /= len(val_loader)
    history['val_loss'].append(epoch_val_loss)
    
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:03d} | Train: {epoch_metrics[0]:.4f} (IQ:{epoch_metrics[1]:.4f}) | Val: {epoch_val_loss:.4f}")
    
    # Early Stopping
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), 'transformer_nle_v2_best.pth')
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print("Early stopping triggered!")
            break


## 11. Loss Component Breakdown Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Total Train/Val
axes[0].plot(history['train_loss'], label='Train Total')
axes[0].plot(history['val_loss'], label='Val Total')
axes[0].set_title('Total Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True)

# Components
axes[1].plot(history['train_mse_iq'], label='IQ MSE')
axes[1].plot(np.array(history['train_mse_amp']) * AMP_WEIGHT, label='Amp Loss (Weighted)')
axes[1].plot(np.array(history['train_snap']) * SNAP_WEIGHT, label='Snap Loss (Weighted)')
axes[1].plot(np.array(history['train_dc']) * DC_WEIGHT, label='DC Loss (Weighted)')
axes[1].set_title('Training Loss Components')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True)

plt.show()


## 12. Inference and Overlapping Window Reconstruction

In [ ]:
model.load_state_dict(torch.load('transformer_nle_v2_best.pth', map_location=device, weights_only=True))
model.eval()

# Reconstruct entire validation sequence for final plotting (since NLE ON is a teacher, we should check validation)
# But to compare directly with previous metrics, we reconstruct the full dataset here, 
# although validation metrics should only be drawn from the validation set conceptually.
# We will reconstruct the full off_data sequence to match shapes.
full_dataset = NLEWindowDataset(off_data_norm, on_data_norm, WINDOW, STRIDE)
full_loader = DataLoader(full_dataset, batch_size=BATCH_SIZE, shuffle=False)

accumulated_preds = np.zeros_like(off_data_norm)
overlap_counts = np.zeros(len(off_data_norm))

idx = 0
with torch.no_grad():
    for x_batch, _ in full_loader:
        x_batch = x_batch.to(device)
        pred_delta_batch = model(x_batch).cpu().numpy()
        
        for i in range(len(pred_delta_batch)):
            start = (idx + i) * STRIDE
            end = start + WINDOW
            
            accumulated_preds[start:end] += pred_delta_batch[i]
            overlap_counts[start:end] += 1
            
        idx += len(pred_delta_batch)

overlap_counts[overlap_counts == 0] = 1
final_delta_hat = accumulated_preds / overlap_counts[:, None]
final_y_hat_norm = off_data_norm + final_delta_hat

# Inverse transform normalization
final_y_hat = final_y_hat_norm * std_val + mean_val

print("Original Input Shape:", off_data.shape)
print("Reconstructed Output Shape:", final_y_hat.shape)
assert off_data.shape == final_y_hat.shape, "Shape mismatch after reconstruction!"


## 13. Amplitude Sanity Check

In [ ]:
def get_amplitudes(data):
    amps = np.sqrt(data[:, 0]**2 + data[:, 1]**2)
    return np.mean(amps), np.sqrt(np.mean(amps**2))

mean_amp_off, rms_amp_off = get_amplitudes(off_data)
mean_amp_ml, rms_amp_ml = get_amplitudes(final_y_hat)
mean_amp_on, rms_amp_on = get_amplitudes(on_data)

print(f"{'Data':<20} | {'Mean Amp':<10} | {'RMS Amp':<10}")
print("-" * 45)
print(f"{'NLE OFF':<20} | {mean_amp_off:<10.4f} | {rms_amp_off:<10.4f}")
print(f"{'Transformer V2':<20} | {mean_amp_ml:<10.4f} | {rms_amp_ml:<10.4f}")
print(f"{'DSP NLE ON':<20} | {mean_amp_on:<10.4f} | {rms_amp_on:<10.4f}")


## 14. Constellation Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot subset for clarity
subset = 5000

def plot_constellation(ax, data, title):
    ax.scatter(data[:subset, 0], data[:subset, 1], s=1, alpha=0.5, color='b')
    ax.set_title(title)
    ax.set_xlabel('I')
    ax.set_ylabel('Q')
    ax.grid(True)
    ax.set_aspect('equal', 'box')
    ax.set_xlim([-3, 3])
    ax.set_ylim([-3, 3])
    ax.axhline(0, color='black', linewidth=0.5)
    ax.axvline(0, color='black', linewidth=0.5)

plot_constellation(axes[0], off_data, "DSP NLE OFF")
plot_constellation(axes[1], final_y_hat, "Transformer ML NLE V2")
plot_constellation(axes[2], on_data, "DSP NLE ON (Reference)")

plt.tight_layout()
plt.show()


## 15. Numerical Cluster Quality & Direct DSP Comparison
Evaluate EVM, direct MSE to DSP target, and Normalized Improvement.

In [ ]:
def calculate_evm(signal, reference):
    error = signal - reference
    evm_rms = np.sqrt(np.mean(np.sum(error**2, axis=1))) / np.sqrt(np.mean(np.sum(reference**2, axis=1)))
    return evm_rms * 100

evm_off = calculate_evm(off_data, on_data)
evm_ml = calculate_evm(final_y_hat, on_data)

mse_off_vs_on = np.mean((off_data - on_data)**2)
mse_ml_vs_on = np.mean((final_y_hat - on_data)**2)

# Normalized Improvement
norm_improvement = mse_ml_vs_on / mse_off_vs_on

print(f"EVM (DSP NLE OFF): {evm_off:.2f}%")
print(f"EVM (Transformer V2): {evm_ml:.2f}%")
print(f"MSE (NLE OFF -> DSP NLE ON): {mse_off_vs_on:.6f}")
print(f"MSE (Transformer -> DSP NLE ON): {mse_ml_vs_on:.6f}")
print(f"Relative Error Ratio (ML/OFF): {norm_improvement:.4f} (Lower is better)")


## 16. BER Calculation
We evaluate the BER across all 4 potential phase mappings to ensure polarity/sign convention doesn't artificially inflate the BER.

In [ ]:
def qpsk_demodulate_all_phases(signal):
    bits = np.zeros((len(signal), 2), dtype=int)
    bits[:, 0] = (signal[:, 0] > 0).astype(int)
    bits[:, 1] = (signal[:, 1] > 0).astype(int)
    return bits

bits_ref = qpsk_demodulate_all_phases(on_data)
bits_off = qpsk_demodulate_all_phases(off_data)
bits_ml = qpsk_demodulate_all_phases(final_y_hat)

ber_off = np.mean(bits_off != bits_ref)
ber_ml = np.mean(bits_ml != bits_ref)

print(f"BER (DSP NLE OFF): {ber_off:.6f}")
print(f"BER (Transformer V2): {ber_ml:.6f}")

if ber_ml > 0.4:
    print("WARNING: BER > 0.4 detected. Investigating potential phase shift...")
    # Rotate ML signal by 90, 180, 270 degrees and check BER
    for phase in [np.pi/2, np.pi, 3*np.pi/2]:
        c, s = np.cos(phase), np.sin(phase)
        rot_matrix = np.array([[c, -s], [s, c]])
        rotated_ml = final_y_hat @ rot_matrix.T
        bits_ml_rot = qpsk_demodulate_all_phases(rotated_ml)
        rot_ber = np.mean(bits_ml_rot != bits_ref)
        print(f"  BER at phase +{np.degrees(phase)}: {rot_ber:.6f}")


## 17. Final Notebook Output

| Model | BER | EVM (%) | MSE to DSP NLE ON |
|---|---|---|---|
| NLE OFF (Distorted) | **{ber_off}** | **{evm_off}** | **{mse_off}** |
| Transformer V2 | **{ber_ml}** | **{evm_ml}** | **{mse_ml}** |
| DSP NLE ON (Reference) | 0.00 | 0.00 | 0.00 |

### Interpretation
- **Did the Transformer improve over NLE OFF?**: *Answer after running (Look for reduced MSE and EVM)*
- **Did it approach DSP NLE ON?**: *Check constellation clustering quality and EVM*
- **Did it improve over the existing MLP baseline?**: *Compare these V2 metrics vs MLP metrics*
- **Is the Transformer worth the additional computational complexity?**: *If MSE_ML ≈ MSE_MLP, then sequence attention may not be offering significant gains over memoryless formulation.*
